# WKNN — Weighted K-Nearest Neighbor

---

## Pengertian

**WKNN (Weighted K-Nearest Neighbor)** merupakan pengembangan dari algoritma K-Nearest Neighbors (KNN) yang menggunakan konsep **pembobotan** pada setiap tetangga terdekat.

Pada metode ini, setiap data tetangga **tidak memberikan pengaruh yang sama**. Data yang memiliki jarak lebih dekat dengan data target akan diberikan **bobot yang lebih besar** dibandingkan data yang jaraknya lebih jauh. Dengan demikian, data yang lebih mirip akan memiliki kontribusi yang lebih besar dalam proses prediksi.

Metode WKNN sering digunakan untuk mengisi nilai yang hilang (**missing value**) dalam dataset.

## Dataset

| No | IPK | PO        | JML |
|----|-----|-----------|-----|
| 1  | 2   | 2.000.000 | 2   |
| 2  | 3   | 3.000.000 | 3   |
| 3  | 4   | 2.000.000 | 2   |
| 4  | 2   | 2.000.000 | 3   |
| 5  | 3   | 3.000.000 | 2   |
| 6  | 4   | 4.000.000 | 3   |
| 7  | 2   | 3.000.000 | **?** |

Objek ke-7 pada kolom **JML** memiliki nilai kosong yang akan diimputasi menggunakan WKNN.

## Langkah 1 — Normalisasi Data (Min-Max)

Sebelum menghitung jarak, semua atribut dinormalisasi menggunakan **Min-Max Normalization** agar berada pada skala yang sama:

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

Hasil normalisasi:

| No | IPK | PO  | JML |
|----|-----|-----|-----|
| 1  | 0   | 0   | 0   |
| 2  | 0.5 | 0.5 | 1   |
| 3  | 1   | 0   | 0   |
| 4  | 0   | 0   | 1   |
| 5  | 0.5 | 0.5 | 0   |
| 6  | 1   | 1   | 1   |
| 7  | 0   | 0.5 | **?** |

## Langkah 2 — Menghitung Kemiripan (Similarity)

Kemiripan antara data target (objek ke-7) dengan setiap tetangga dihitung menggunakan rumus:

$$\frac{1}{s_i} = \sum_{h \in O_i \cap O_j} (y_{ih} - y_{jh})^2$$

**Keterangan:**
- $(y_{ih} - y_{jh})^2$ : kuadrat selisih nilai atribut data target dengan tetangga
- $O_i \cap O_j$ : hanya atribut yang tersedia (tidak kosong) di **kedua** baris
- $s_i$ : bobot — semakin kecil jarak, semakin besar bobot (semakin mirip)

> Karena JML objek ke-7 kosong, perhitungan hanya menggunakan **IPK** dan **PO**.

---

**Objek 1** — target: (IPK=0, PO=0.5) vs tetangga: (IPK=0, PO=0)

$$\frac{1}{s_1} = (0-0)^2 + (0.5-0)^2 = 0 + 0.25 = 0.25 \quad \Rightarrow \quad s_1 = \frac{1}{0.25} = 4$$

**Objek 2** — target: (IPK=0, PO=0.5) vs tetangga: (IPK=0.5, PO=0.5)

$$\frac{1}{s_2} = (0-0.5)^2 + (0.5-0.5)^2 = 0.25 + 0 = 0.25 \quad \Rightarrow \quad s_2 = \frac{1}{0.25} = 4$$

**Objek 3** — target: (IPK=0, PO=0.5) vs tetangga: (IPK=1, PO=0)

$$\frac{1}{s_3} = (0-1)^2 + (0.5-0)^2 = 1 + 0.25 = 1.25 \quad \Rightarrow \quad s_3 = \frac{1}{1.25} = 0.8$$

**Objek 4** — target: (IPK=0, PO=0.5) vs tetangga: (IPK=0, PO=0)

$$\frac{1}{s_4} = (0-0)^2 + (0.5-0)^2 = 0 + 0.25 = 0.25 \quad \Rightarrow \quad s_4 = \frac{1}{0.25} = 4$$

**Objek 5** — target: (IPK=0, PO=0.5) vs tetangga: (IPK=0.5, PO=0.5)

$$\frac{1}{s_5} = (0-0.5)^2 + (0.5-0.5)^2 = 0.25 + 0 = 0.25 \quad \Rightarrow \quad s_5 = \frac{1}{0.25} = 4$$

**Objek 6** — target: (IPK=0, PO=0.5) vs tetangga: (IPK=1, PO=1)

$$\frac{1}{s_6} = (0-1)^2 + (0.5-1)^2 = 1 + 0.25 = 1.25 \quad \Rightarrow \quad s_6 = \frac{1}{1.25} = 0.8$$

---

**Ringkasan Bobot:**

| Objek | $1/s_i$ | $s_i$ |
|-------|---------|-------|
| 1     | 0.25    | 4.0   |
| 2     | 0.25    | 4.0   |
| 3     | 1.25    | 0.8   |
| 4     | 0.25    | 4.0   |
| 5     | 0.25    | 4.0   |
| 6     | 1.25    | 0.8   |

## Langkah 3 — Imputasi WKNN

Nilai missing value diisi menggunakan **rata-rata tertimbang**:

$$\hat{y}_{ih} = \frac{\sum_{j \in IK_{ih}} s_i(y_j) \cdot y_{jh}}{\sum_{j \in IK_{ih}} s_i(y_j)}$$

**Keterangan:**
- $\hat{y}_{ih}$ : nilai prediksi untuk baris $i$ pada atribut $h$
- $IK_{ih}$ : himpunan K tetangga terdekat yang memiliki nilai pada atribut $h$
- $s_i(y_j) \cdot y_{jh}$ : nilai tetangga dikalikan bobotnya
- $\sum s_i$ : normalisasi bobot (menghasilkan rata-rata tertimbang)

Menerapkan rumus (K=6, semua tetangga digunakan):

$$\hat{y}_{7,JML} = \frac{(0 \times 4) + (1 \times 4) + (0 \times 0.8) + (1 \times 4) + (0 \times 4) + (1 \times 0.8)}{4 + 4 + 0.8 + 4 + 4 + 0.8}$$

$$= \frac{0 + 4 + 0 + 4 + 0 + 0.8}{17.6} = \frac{8.8}{17.6} = \mathbf{0.5}$$

Nilai prediksi JML untuk objek ke-7 (ternormalisasi) = **0.5**

## Implementasi Python — Manual

In [ ]:
import pandas as pd
import numpy as np

data = {
    'IPK': [2, 3, 4, 2, 3, 4, 2],
    'PO':  [2000000, 3000000, 2000000, 2000000, 3000000, 4000000, 3000000],
    'JML': [2, 3, 2, 3, 2, 3, np.nan]
}
df = pd.DataFrame(data, index=range(1, 8))
df.index.name = 'No'

print("=== Dataset Asli ===")
print(df)

In [ ]:
# 2. Normalisasi Min-Max Manual

df_norm = df.copy()

# Gunakan min/max dari data training (baris 1-6) sebagai acuan
for col in ['IPK', 'PO', 'JML']:
    ref = df[col].iloc[:6]
    x_min = ref.min()
    x_max = ref.max()
    df_norm[col] = (df[col] - x_min) / (x_max - x_min)

print("=== Data Setelah Normalisasi Min-Max ===")
print(df_norm)

In [ ]:
# 3. Menghitung Kemiripan (Similarity)
target = df_norm.iloc[6]          
fitur  = ['IPK', 'PO']            

print("=== Perhitungan Kemiripan ===")
print(f"Data Target (Objek 7): IPK={target['IPK']}, PO={target['PO']}, JML=?\n")

bobot = []
for i in range(6):
    tetangga = df_norm.iloc[i]
    inv_si   = sum((target[f] - tetangga[f])**2 for f in fitur)
    si       = 1 / inv_si if inv_si != 0 else np.inf
    bobot.append(si)

    print(f"Objek {i+1}:")
    for f in fitur:
        print(f"  ({target[f]} - {tetangga[f]})² = {(target[f]-tetangga[f])**2:.4f}")
    print(f"  1/s_{i+1} = {inv_si:.4f}  →  s_{i+1} = {si:.4f}\n")

In [ ]:
# 4. Imputasi WKNN Manual

jml_tetangga = df_norm['JML'].iloc[:6].values

pembilang  = sum(bobot[j] * jml_tetangga[j] for j in range(6))
penyebut   = sum(bobot)
jml_pred   = pembilang / penyebut

print("=== Proses Imputasi WKNN ===")
print("\nPerhitungan pembilang (s_i × JML_j):")
for j in range(6):
    print(f"  Objek {j+1}: {bobot[j]:.4f} × {jml_tetangga[j]:.1f} = {bobot[j]*jml_tetangga[j]:.4f}")

print(f"\nTotal pembilang : {pembilang:.4f}")
print(f"Total penyebut  : {penyebut:.4f}")
print(f"\nNilai JML (ternormalisasi) = {pembilang:.4f} / {penyebut:.4f} = {jml_pred:.4f}")

# Kembalikan ke skala asli
jml_min  = df['JML'].min()   
jml_max  = df['JML'].max()   
jml_asli = jml_pred * (jml_max - jml_min) + jml_min
print(f"Nilai JML (skala asli)     = {jml_asli:.4f}")

## Implementasi Python — Menggunakan sklearn

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor

df_train = df.iloc[:6].copy()
df_test  = df.iloc[6:].copy()

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

train_X = scaler_X.fit_transform(df_train[['IPK', 'PO']])
test_X  = scaler_X.transform(df_test[['IPK', 'PO']])
train_y = scaler_y.fit_transform(df_train[['JML']])

def wknn_weights(distances):
    return 1 / (distances**2)

knn = KNeighborsRegressor(n_neighbors=6, weights=wknn_weights, metric='euclidean')
knn.fit(train_X, train_y)

pred_norm = knn.predict(test_X)
pred_asli = scaler_y.inverse_transform(pred_norm)

print("=== Hasil Prediksi WKNN (sklearn) ===")
print(f"Prediksi JML (ternormalisasi) : {pred_norm[0][0]:.4f}")
print(f"Prediksi JML (skala asli)     : {pred_asli[0][0]:.4f}")

## Kesimpulan

- WKNN mengisi missing value dengan **rata-rata tertimbang** dari K tetangga terdekat
- Tetangga yang lebih dekat mendapat **bobot lebih besar** ($s_i = 1/d^2$), sehingga lebih berpengaruh
- Data harus dinormalisasi terlebih dahulu agar jarak antar atribut dapat dibandingkan secara adil
- Pada contoh ini, nilai prediksi JML untuk objek ke-7 = **0.5** (ternormalisasi)